## WP001 — Walk-Forward Cross-Validation Baseline

See `README.md` in this folder for methodology, results, and metric explanations. This notebook only runs the CV pipeline: data loading, the rolling-window walk-forward CV loop (each window in its own subprocess — see `scripts/run_cv_window.py`), and evaluation (MAE, pooled RPS, calibration, bootstrap significance).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append('/users/hadiahmed/documents/projects/football-predictor/src')

In [ ]:
from football_model.data.get_data import get_understat_data
from football_model.features.add_metadata import add_rounds_to_data, add_home_away_goals_xg, add_match_ids

In [20]:
# Rolling window cross-validation
import pandas as pd
from scipy.stats import poisson
import pymc as pm
import numpy as np

# Get full dataset — two seasons, spanning a real relegation/promotion boundary,
# so the CV windows exercise the multi-season handling (relegation freeze,
# season-start variance inflation) rather than just a single season.
df_cv = get_understat_data(leagues=['EPL'], years=[str(i) for i in range(2020,2026)])
df_cv = add_rounds_to_data(df_cv)
df_cv = add_match_ids(df_cv)
df_cv = add_home_away_goals_xg(df_cv)

max_round_cv = df_cv['round'].max()
print(f"Total rounds available: {max_round_cv}")

# Configuration — start training a few rounds into the second season so every
# window straddles the relegation/promotion boundary, and step by more than 1
# round so this stays a reasonable number of full NUTS runs. Tune step_size
# down (more windows, slower) or up (fewer windows, faster) as needed.
seasons_sorted = sorted(df_cv['season'].unique())
season1_end = df_cv.loc[df_cv['season'] == seasons_sorted[0], 'round'].max()
min_train_rounds = season1_end + 3
test_window = 1
step_size = 5

# Define windows
windows = []
current_train_end = min_train_rounds
while current_train_end + test_window <= max_round_cv:
    test_start = current_train_end + 1
    test_end = min(test_start + test_window - 1, max_round_cv)
    windows.append({
        'train_start': 1,
        'train_end': current_train_end,
        'test_start': test_start,
        'test_end': test_end
    })
    current_train_end += step_size

print(f"\n=== CROSS-VALIDATION WINDOWS ===")
for i, w in enumerate(windows, 1):
    n_train = df_cv[df_cv['round'] <= w['train_end']]['match_id'].nunique()
    n_test = df_cv[(df_cv['round'] >= w['test_start']) & 
                   (df_cv['round'] <= w['test_end'])]['match_id'].nunique()
    print(f"Window {i}: Train rounds {w['train_start']}-{w['train_end']} ({n_train} matches) → "
          f"Test rounds {w['test_start']}-{w['test_end']} ({n_test} matches)")

Total rounds available: 208

=== CROSS-VALIDATION WINDOWS ===
Window 1: Train rounds 1-36 (410 matches) → Test rounds 37-37 (10 matches)
Window 2: Train rounds 1-41 (460 matches) → Test rounds 42-42 (10 matches)
Window 3: Train rounds 1-46 (509 matches) → Test rounds 47-47 (20 matches)
Window 4: Train rounds 1-51 (571 matches) → Test rounds 52-52 (10 matches)
Window 5: Train rounds 1-56 (624 matches) → Test rounds 57-57 (12 matches)
Window 6: Train rounds 1-61 (678 matches) → Test rounds 62-62 (11 matches)
Window 7: Train rounds 1-66 (732 matches) → Test rounds 67-67 (14 matches)
Window 8: Train rounds 1-71 (790 matches) → Test rounds 72-72 (10 matches)
Window 9: Train rounds 1-76 (847 matches) → Test rounds 77-77 (10 matches)
Window 10: Train rounds 1-81 (906 matches) → Test rounds 82-82 (7 matches)
Window 11: Train rounds 1-86 (959 matches) → Test rounds 87-87 (10 matches)
Window 12: Train rounds 1-91 (1011 matches) → Test rounds 92-92 (10 matches)
Window 13: Train rounds 1-96 (1066 

/Users/hadiahmed/Documents/projects/football-predictor/src/football_model/features/add_metadata.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('season', group_keys=False).apply(adjust_round)
/Users/hadiahmed/Documents/projects/football-predictor/src/football_model/features/add_metadata.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('season', group_keys=False).apply(make_rou

In [ ]:
# Run cross-validation — each window in its OWN subprocess (scripts/run_cv_window.py),
# so nothing (JAX compiled programs, thread state, memory) can accumulate across
# windows the way it did when all 35+ ran sequentially inside this one kernel.
# A hung/killed window just gets retried on the next run of this cell.
import pickle
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path('/Users/hadiahmed/Documents/projects/football-predictor')
WP_DIR = REPO_ROOT / 'work_products' / 'wp001_walkforward_cv_baseline'
data_path = WP_DIR / 'cv_shared_data.pkl'
checkpoint_path = WP_DIR / 'cv_checkpoint.pkl'
script_path = REPO_ROOT / 'scripts' / 'run_cv_window.py'

WINDOW_TIMEOUT_SECONDS = 1200  # 20 min/window — a window that hangs even in its
                                # own fresh process gets killed and skipped rather
                                # than blocking the whole sweep forever.

# Save the shared inputs once — every subprocess loads from here instead of
# re-fetching/recomputing df_cv.
with open(data_path, 'wb') as f:
    pickle.dump({'df_cv': df_cv, 'windows': windows}, f)

def load_checkpoint():
    if checkpoint_path.exists():
        with open(checkpoint_path, 'rb') as f:
            return pickle.load(f)
    return {'results': [], 'cv_match_predictions': []}

completed = {r['window'] for r in load_checkpoint()['results']}
print(f"{len(completed)}/{len(windows)} windows already completed.")

for i in range(1, len(windows) + 1):
    if i in completed:
        continue
    print(f"\n{'='*60}\nWINDOW {i}/{len(windows)}\n{'='*60}")
    try:
        subprocess.run(
            [sys.executable, str(script_path),
             '--data-path', str(data_path),
             '--checkpoint-path', str(checkpoint_path),
             '--window-index', str(i)],
            timeout=WINDOW_TIMEOUT_SECONDS,
            check=True,
        )
    except subprocess.TimeoutExpired:
        print(f"WINDOW {i} exceeded {WINDOW_TIMEOUT_SECONDS}s — killed and skipped. "
              "Re-run this cell to retry it (it's not marked complete in the checkpoint).")
    except subprocess.CalledProcessError:
        print(f"WINDOW {i} failed with a real error (see traceback above) — skipped. "
              "Re-run this cell to retry it.")

# Load the final checkpoint into the variable names the rest of the notebook
# expects (results_df below, and the pooled RPS/calibration cell after it).
final_checkpoint = load_checkpoint()
results = sorted(final_checkpoint['results'], key=lambda r: r['window'])
cv_match_predictions = final_checkpoint['cv_match_predictions']

print(f"\n{'='*60}")
print(f"CROSS-VALIDATION COMPLETE: {len(results)}/{len(windows)} windows finished")
print(f"{'='*60}")

In [22]:
# Summarize cross-validation results
results_df = pd.DataFrame(results)

print("\n=== CROSS-VALIDATION SUMMARY ===\n")
print(results_df.to_string(index=False))

print(f"\n{'='*60}")
print("AVERAGE PERFORMANCE ACROSS ALL WINDOWS")
print(f"{'='*60}")
print(f"Mean MAE: {results_df['mae'].mean():.3f} ± {results_df['mae'].std():.3f}")
print(f"Mean Log Likelihood: {results_df['log_likelihood'].mean():.2f} ± {results_df['log_likelihood'].std():.2f}")
print(f"Mean LL Improvement over Naive: {results_df['ll_improvement'].mean():.2f} ± {results_df['ll_improvement'].std():.2f}")

print(f"\n{'='*60}")
print("PERFORMANCE STABILITY")
print(f"{'='*60}")
print(f"MAE Range: {results_df['mae'].min():.3f} to {results_df['mae'].max():.3f}")
print(f"LL Improvement Range: {results_df['ll_improvement'].min():.2f} to {results_df['ll_improvement'].max():.2f}")
print(f"All windows beat naive baseline: {(results_df['ll_improvement'] > 0).all()}")


=== CROSS-VALIDATION SUMMARY ===

 window train_rounds test_rounds  n_train  n_test      mae  log_likelihood   ll_naive  ll_improvement
      1         1-36       37-37      410      20 1.138836      -30.068300 -31.143097        1.074798
      2         1-41       42-42      460      20 1.297122      -40.690205 -40.901633        0.211429
      3         1-46       47-47      509      40 0.684258      -51.941739 -55.765140        3.823401
      4         1-51       52-52      571      20 0.763485      -28.927745 -29.652049        0.724304
      5         1-56       57-57      624      24 1.050245      -35.326815 -39.447747        4.120932
      6         1-61       62-62      678      22 1.167306      -37.614803 -37.641336        0.026533
      7         1-66       67-67      732      28 0.922828      -41.683639 -45.352456        3.668817
      8         1-71       72-72      790      20 0.941283      -32.555295 -36.687125        4.131830
      9         1-76       77-77      847      

In [ ]:
# Evaluation utilities (RPS, calibration, bootstrap CI) — see README for what
# each one means and why MAE/log-likelihood alone aren't enough.
def outcome_probs_from_lambda(lambda_home, lambda_away, max_goals=10):
    """Home/draw/away win probabilities from Poisson goal-rate parameters,
    via exact convolution over the scoreline grid (not Monte Carlo)."""
    goals = np.arange(max_goals + 1)
    ph = poisson.pmf(goals, lambda_home)
    pa = poisson.pmf(goals, lambda_away)
    grid = np.outer(ph, pa)  # grid[i, j] = P(home scores i, away scores j)
    p_home = np.tril(grid, -1).sum()
    p_draw = np.trace(grid)
    p_away = np.triu(grid, 1).sum()
    total = p_home + p_draw + p_away  # <1 due to truncation at max_goals; renormalize
    return p_home / total, p_draw / total, p_away / total


def match_outcome(goals_home, goals_away):
    if goals_home > goals_away:
        return 'H'
    if goals_home == goals_away:
        return 'D'
    return 'A'


def rps_home_draw_away(p_home, p_draw, p_away, actual):
    """Ranked Probability Score for a single match (lower is better, 0=perfect)."""
    cp1, cp2 = p_home, p_home + p_draw
    ce1 = 1.0 if actual == 'H' else 0.0
    ce2 = 1.0 if actual in ('H', 'D') else 0.0
    return 0.5 * ((cp1 - ce1) ** 2 + (cp2 - ce2) ** 2)


def reliability_table(pred_probs, actual_flags, n_bins=5):
    """Calibration check: bucket predictions by predicted probability and
    compare to the actual frequency of the event within each bucket."""
    pred_probs = np.asarray(pred_probs)
    actual_flags = np.asarray(actual_flags)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_idx = np.clip(np.digitize(pred_probs, bins[1:-1]), 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        rows.append({
            'bin': f"{bins[b]:.1f}-{bins[b+1]:.1f}",
            'n_matches': n,
            'mean_predicted': pred_probs[mask].mean(),
            'actual_frequency': actual_flags[mask].mean(),
        })
    return pd.DataFrame(rows)


def bootstrap_mean_ci(values, n_boot=5000, alpha=0.05, seed=0):
    """Bootstrap confidence interval for the mean of `values`."""
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    n = len(values)
    boot_means = np.array([
        rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)
    ])
    lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return values.mean(), lo, hi

In [ ]:
# Is the average CV improvement actually distinguishable from zero, or is it
# noise? This works with your CURRENT results_df as-is — no rerun needed.
mean_imp, lo, hi = bootstrap_mean_ci(results_df['ll_improvement'].values, seed=0)
print(f"Windows: {len(results_df)}")
print(f"Mean LL improvement over naive: {mean_imp:.2f}")
print(f"95% bootstrap CI: [{lo:.2f}, {hi:.2f}]")
if lo > 0:
    print("→ CI excludes zero: the improvement is likely real at this sample size, not noise.")
elif hi < 0:
    print("→ CI excludes zero (negative side): the model is likely worse than naive here.")
else:
    print("→ CI includes zero: can't yet distinguish this from no improvement.")

In [ ]:
# Pool ALL match-level predictions across every CV window (not just a single
# holdout) for a properly-powered RPS + calibration check.
if 'cv_match_predictions' in dir() and len(cv_match_predictions) > 0:
    # Naive baseline: league-average goals across the whole CV dataset —
    # matches the "naive_lambda" used inside run_cv_window.py per window,
    # just computed once here for the pooled comparison.
    historical_avg_home = df_cv['goals_home'].mean()
    historical_avg_away = df_cv['goals_away'].mean()

    cv_pred_df = pd.DataFrame(cv_match_predictions)
    cv_pred_df['outcome'] = [
        match_outcome(r.goals_home, r.goals_away) for r in cv_pred_df.itertuples()
    ]

    cv_probs = [outcome_probs_from_lambda(r.lambda_home, r.lambda_away) for r in cv_pred_df.itertuples()]
    cv_rps = np.mean([rps_home_draw_away(*p, a) for p, a in zip(cv_probs, cv_pred_df['outcome'])])

    naive_probs_cv = [outcome_probs_from_lambda(historical_avg_home, historical_avg_away)] * len(cv_pred_df)
    naive_rps_cv = np.mean([rps_home_draw_away(*p, a) for p, a in zip(naive_probs_cv, cv_pred_df['outcome'])])

    print(f"Pooled across {len(cv_pred_df)} matches from {cv_pred_df['window'].nunique()} CV windows\n")
    print(f"Pooled Model RPS: {cv_rps:.4f}")
    print(f"Pooled Naive RPS: {naive_rps_cv:.4f}")

    cv_p_home = np.array([p[0] for p in cv_probs])
    cv_actual_home = (cv_pred_df['outcome'] == 'H').astype(int).values

    print("\n=== POOLED CALIBRATION: predicted P(home win) vs actual home-win rate ===")
    print(reliability_table(cv_p_home, cv_actual_home, n_bins=8).to_string(index=False))

    # With this many matches, a bootstrap CI on match-level RPS is also meaningful
    rps_diff = np.array([
        rps_home_draw_away(*p_m, a) - rps_home_draw_away(*p_n, a)
        for p_m, p_n, a in zip(cv_probs, naive_probs_cv, cv_pred_df['outcome'])
    ])
    mean_rps_diff, lo_r, hi_r = bootstrap_mean_ci(-rps_diff)  # negate: RPS lower=better, so flip sign to "improvement"
    print(f"\nMean RPS improvement over naive: {mean_rps_diff:.4f}, 95% CI: [{lo_r:.4f}, {hi_r:.4f}]")
else:
    print("cv_match_predictions not found or empty — re-run the CV loop cell above "
          "so it gets populated, then run this cell again.")